# MJ AI — first LoRA training (fixed + safer export)\nThis notebook fine-tunes Qwen2.5-0.5B-Instruct with an MJ AI adapter. It does **not** train a foundation model from scratch.\n\n**Important:** the ZIP/export cell only works after training completes successfully and `adapter_config.json` exists. The starter dataset is only a pipeline test; use a larger reviewed dataset for meaningful customization.

In [ ]:
!pip -q install -U transformers datasets peft accelerate\nimport os, subprocess\nrepo_dir = '/content/Ai-video-free'\nif os.path.isdir(repo_dir):\n    subprocess.run(['git', '-C', repo_dir, 'pull', '--ff-only'], check=True)\nelse:\n    subprocess.run(['git', 'clone', 'https://github.com/singwithsashu-a11y/Ai-video-free.git', repo_dir], check=True)\nprint('Repository ready:', repo_dir)

In [ ]:
%cd /content/Ai-video-free\nimport torch, os\nprint('PyTorch:', torch.__version__)\nprint('CUDA available:', torch.cuda.is_available())\nif torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))\nelse: print('WARNING: no GPU detected; training may be slow or run out of memory.')

## 1. Choose training data\nDefault is the tiny repository seed dataset, used only to test the pipeline. For better results, upload a high-quality JSONL dataset you own or have permission to use. Do not include passwords, private chats, or personal/sensitive data.

In [ ]:
import os\nos.makedirs('/content/mj-ai-data', exist_ok=True)\nraw_path = '/content/Ai-video-free/training/seed_data.jsonl'\nif not os.path.isfile(raw_path):\n    raise FileNotFoundError(f'Seed dataset not found: {raw_path}. Re-run the repository setup cell.')\nprint('Selected starter dataset:', raw_path)\nprint('To use your own data, run the optional upload cell below, then continue.')

In [ ]:
# OPTIONAL: upload your own .jsonl file. Cancel/skip the file picker to keep the starter dataset.\nfrom google.colab import files\nuploaded = files.upload()\nif uploaded:\n    name = next(iter(uploaded))\n    if not name.lower().endswith('.jsonl'):\n        raise ValueError('Please upload a .jsonl file')\n    raw_path = '/content/mj-ai-data/custom.jsonl'\n    with open(raw_path, 'wb') as f: f.write(uploaded[name])\nprint('Training data selected:', raw_path)

In [ ]:
import subprocess, os\nprocessed_path = '/content/mj-ai-data/processed.jsonl'\nsubprocess.run(['python', '/content/Ai-video-free/training/prepare_dataset.py', raw_path, processed_path], check=True)\nwith open(processed_path, encoding='utf-8') as f: examples = sum(1 for line in f if line.strip())\nprint('Validated examples:', examples)\nif examples == 0: raise RuntimeError('No valid examples found. Check your JSONL format.')

## 2. Train the adapter\nRun this cell and wait until it prints `SUCCESS`. If training raises an error, fix that error first—do not run the ZIP cell yet. Training may take time and free GPU availability is not guaranteed.

In [ ]:
import subprocess, os\noutput_dir = '/content/mj-ai-lora'\nzip_path = '/content/mj-ai-lora.zip'\n# Clear an incomplete prior output so the export cannot accidentally package stale files.\nif os.path.isdir(output_dir):\n    import shutil; shutil.rmtree(output_dir)\nif os.path.exists(zip_path): os.remove(zip_path)\ncommand = [\n    'python', '/content/Ai-video-free/training/train_lora.py',\n    '--model', 'Qwen/Qwen2.5-0.5B-Instruct',\n    '--data', processed_path, '--output', output_dir,\n    '--epochs', '3', '--max-length', '512'\n]\nprint('Starting:', ' '.join(command), flush=True)\nsubprocess.run(command, check=True)\nadapter_config = os.path.join(output_dir, 'adapter_config.json')\nif not os.path.isfile(adapter_config):\n    raise RuntimeError('Training command finished but adapter_config.json is missing. Do not export; inspect the training logs.')\nadapter_files = os.listdir(output_dir)\nprint('TRAINING VERIFIED. Adapter files:', adapter_files)

## 3. Create and download the ZIP\nThis cell checks the trained adapter before packaging. If it says no adapter was found, go back and successfully complete the training cell first.

In [ ]:
import os, zipfile\nfrom google.colab import files\noutput_dir = '/content/mj-ai-lora'\nzip_path = '/content/mj-ai-lora.zip'\nconfig_path = os.path.join(output_dir, 'adapter_config.json')\nif not os.path.isfile(config_path):\n    raise FileNotFoundError('MJ AI adapter not found: ' + config_path + '. Training did not finish successfully; review the previous cell output and rerun training.')\nadapter_weight_files = [n for n in os.listdir(output_dir) if n.endswith(('.safetensors', '.bin'))]\nif not adapter_weight_files:\n    raise FileNotFoundError('Adapter config exists but no adapter weight file (.safetensors/.bin) was found. Do not download an empty ZIP.')\nif os.path.exists(zip_path): os.remove(zip_path)\nwith zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:\n    for root, dirs, filenames in os.walk(output_dir):\n        for filename in filenames:\n            full_path = os.path.join(root, filename)\n            zf.write(full_path, os.path.relpath(full_path, output_dir))\nif not os.path.isfile(zip_path) or os.path.getsize(zip_path) == 0:\n    raise RuntimeError('ZIP creation failed or created an empty file.')\nprint('ZIP ready:', zip_path)\nprint('Size:', round(os.path.getsize(zip_path) / 1024 / 1024, 2), 'MB')\nprint('Included files:', zipfile.ZipFile(zip_path).namelist())\nfiles.download(zip_path)

## After download\nThe ZIP contains the **LoRA adapter**, not the full base model. It still requires `Qwen/Qwen2.5-0.5B-Instruct` when running chat. Example:\n\n`python model/mj_ai_chat.py --model Qwen/Qwen2.5-0.5B-Instruct --adapter /path/to/mj-ai-lora`\n\nThe tiny starter dataset only verifies the training pipeline and will not create a strong assistant. Keep the adapter and your data backed up before the temporary Colab session ends.